In [0]:
from pyspark.sql import functions as F

In [0]:
customers = spark.table("bronze.customers")

bad_email_customers = (
    customers
    .limit(5)
    .withColumn("email",F.lit("invalid-email"))
)

bad_country_customers = (
    customers
    .filter(F.col("customer_id").between("C000006", "C000008"))
    .withColumn(
        "country",
        F.lit("Mars")
    )
)

bad_country_customers = (
    customers
    .filter(F.col("customer_id").between("C000006", "C000008"))
    .withColumn(
        "country",
        F.lit("Mars")
    )
)

bad_customers_id =  (
    customers
    .filter(F.col('customer_id') == 'C000009')
    .withColumn('customer_id', F.lit(None).cast('string'))
)

In [0]:
bad_customers = (
    bad_email_customers
    .unionByName(bad_country_customers)
    .unionByName(bad_customers_id)
)

In [0]:
bad_customer_ids = [
    "C000001",
    "C000002",
    "C000003",
    "C000004",
    "C000005",
    "C000006",
    "C000007",
    "C000008",
    "C000009"
]

clean_customers = customers.filter(
    ~F.col("customer_id").isin(bad_customer_ids)
)

customers_test = (
    clean_customers
    .unionByName(bad_customers)
)

print(
    "Original:",
    customers.count()
)

print(
    "Test dataset:",
    customers_test.count()
)

In [0]:
(
    customers_test.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze.customers")
)